# Demonstração gerencial — análise realista com o Random Forest final

Esta versão preserva o formato didático da demonstração gerencial, mas substitui perfis artificiais por **grupos que realmente existem no conjunto oficial de teste**.

Cada teste possui duas camadas:

1. **Resultado técnico:** tamanho dos grupos, alfabetização observada e estimativa média do modelo;
2. **Interpretação para tomada de decisão:** tradução dos resultados para linguagem gerencial.

> As comparações são descritivas e preditivas. Elas mostram associações presentes na base, mas não demonstram que uma característica seja a causa da alfabetização.

## Menu

1. [Preparação e validação dos artefatos](#Etapa-1)
2. [Como os grupos reais são construídos](#Etapa-2)
3. [Teste 1 — Cobertura municipal de energia](#Teste-1)
4. [Teste 2 — Cobertura municipal de conectividade](#Teste-2)
5. [Teste 3 — Cobertura municipal de água e esgoto](#Teste-3)
6. [Teste 4 — Cobertura municipal de biblioteca](#Teste-4)
7. [Teste 5 — Cobertura municipal de laboratório](#Teste-5)
8. [Teste 6 — Cobertura municipal de quadra](#Teste-6)
9. [Teste 7 — Índice agregado de infraestrutura](#Teste-7)
10. [Teste 8 — INSE médio](#Teste-8)
11. [Teste 9 — Contextos observados em SP e CE](#Teste-9)
12. [Teste 10 — Redes estadual e municipal](#Teste-10)
13. [Resumo executivo](#Resumo)
14. [Análise final sobre o peso da UF](#Analise-final-UF)
15. [Guia de interpretação](#Guia-final)

## Etapa 1

### Preparação e validação dos artefatos

O notebook carrega o modelo, o manifesto, as métricas oficiais e a base publicada pelo pipeline completo. Nenhum treinamento ou ajuste do modelo é realizado aqui.

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import polars as pl


def localizar_raiz(inicio=Path.cwd()):
    for candidato in [inicio, *inicio.parents]:
        manifesto = candidato / 'dados' / 'dados_processados' / 'preparado' / 'manifesto_base_modelagem_com_chave_escola.json'
        resultado = candidato / 'resultados' / 'metricas' / 'fase_7' / 'resultado_final.json'
        modelo = candidato / 'modelos' / 'random_forest' / 'modelo_pipeline.joblib'
        if manifesto.exists() and resultado.exists() and modelo.exists():
            return candidato
    raise FileNotFoundError('Artefatos oficiais do pipeline completo não foram encontrados.')


PROJECT_ROOT = localizar_raiz()
MANIFESTO_PATH = PROJECT_ROOT / 'dados' / 'dados_processados' / 'preparado' / 'manifesto_base_modelagem_com_chave_escola.json'
RESULTADO_FINAL_PATH = PROJECT_ROOT / 'resultados' / 'metricas' / 'fase_7' / 'resultado_final.json'
MODELO_PATH = PROJECT_ROOT / 'modelos' / 'random_forest' / 'modelo_pipeline.joblib'

contrato = json.loads(MANIFESTO_PATH.read_text(encoding='utf-8'))
avaliacao_oficial = json.loads(RESULTADO_FINAL_PATH.read_text(encoding='utf-8'))
modelo = joblib.load(MODELO_PATH)

FEATURES = [str(c) for c in avaliacao_oficial['features']]
TARGET = contrato['target']
SPLIT = contrato['controle_split']
LIMIAR = float(avaliacao_oficial['limiar_classificacao'])
METRICAS_OFICIAIS = avaliacao_oficial['metricas_teste']

assert FEATURES == [str(c) for c in contrato['features_modelagem']]
assert modelo.named_steps['preprocessor'].n_features_in_ == len(FEATURES)
assert list(modelo.named_steps['model'].classes_) == [0, 1]

parametros_modelo = modelo.named_steps['model'].get_params()
assert all(parametros_modelo[k] == v for k, v in avaliacao_oficial['parametros'].items())

data_glob = PROJECT_ROOT / Path(contrato['padrao_leitura'])
base = pl.scan_parquet(str(data_glob), hive_partitioning=True)
desenvolvimento = base.filter(pl.col(SPLIT).is_in(['treino', 'validacao']))
teste = base.filter(pl.col(SPLIT) == 'teste').select(FEATURES + [TARGET]).collect()

probabilidades = modelo.predict_proba(teste.select(FEATURES))[:, 1]
predicoes = np.asarray(probabilidades >= LIMIAR, dtype=np.int8)
avaliacao = teste.with_columns(
    pl.Series('probabilidade_modelo', probabilidades),
    pl.Series('classe_modelo', predicoes),
)

assert avaliacao.height == int(avaliacao_oficial['registros_teste'])
matriz = METRICAS_OFICIAIS['matriz_confusao']
acuracia_recalculada = (matriz[0][0] + matriz[1][1]) / sum(map(sum, matriz))
assert np.isclose(acuracia_recalculada, METRICAS_OFICIAIS['accuracy'])

print(f'Modelo oficial: {MODELO_PATH}')
print(f'Registros reais de teste: {avaliacao.height:,}')
print(f'Features validadas: {len(FEATURES)}')
print(f'Acurácia oficial: {METRICAS_OFICIAIS["accuracy"]:.1%}')
print(f'AUC-ROC oficial: {METRICAS_OFICIAIS["roc_auc"]:.1%}')

Modelo oficial: /workspace/modelos/random_forest/modelo_pipeline.joblib
Registros reais de teste: 581,591
Features validadas: 18
Acurácia oficial: 62.1%
AUC-ROC oficial: 67.0%


## Etapa 2

### Como os grupos reais são construídos

As variáveis de infraestrutura representam **proporções municipais de escolas que possuem cada recurso**, variando de 0 a 1. Portanto, não significam que uma escola ou aluno específico tenha ou não o recurso.

Para infraestrutura e INSE:

- os limites dos grupos são definidos no conjunto de desenvolvimento;
- o grupo de menor cobertura contém registros abaixo do percentil 25;
- o grupo de maior cobertura contém registros acima do percentil 75;
- a comparação é feita somente com registros reais do teste oficial.

Para UF e rede, são utilizados diretamente os grupos observados no teste. Os grupos não são idênticos nas demais características; por isso, as diferenças são associações descritivas, não efeitos causais.

In [2]:
INFRAESTRUTURA = [
    'in_biblioteca_sala_leitura', 'in_laboratorio_informatica',
    'in_quadra_esportes', 'in_agua_rede_publica',
    'in_energia_rede_publica', 'in_esgoto_rede_publica',
    'in_internet', 'in_banda_larga',
]

COLUNAS_COMPOSTAS = {
    'indice_conectividade': ['in_internet', 'in_banda_larga'],
    'indice_saneamento': ['in_agua_rede_publica', 'in_esgoto_rede_publica'],
    'indice_infraestrutura': INFRAESTRUTURA,
}


def adicionar_indices(frame):
    return frame.with_columns([
        pl.mean_horizontal([pl.col(c) for c in colunas]).alias(nome)
        for nome, colunas in COLUNAS_COMPOSTAS.items()
    ])


avaliacao = adicionar_indices(avaliacao)
desenvolvimento_contexto = adicionar_indices(
    desenvolvimento.select(INFRAESTRUTURA + ['media_inse'])
)

COLUNAS_FAIXAS = {
    'energia': 'in_energia_rede_publica',
    'conectividade': 'indice_conectividade',
    'saneamento': 'indice_saneamento',
    'biblioteca': 'in_biblioteca_sala_leitura',
    'laboratorio': 'in_laboratorio_informatica',
    'quadra': 'in_quadra_esportes',
    'infraestrutura': 'indice_infraestrutura',
    'inse': 'media_inse',
}

cortes = {}
for nome, coluna in COLUNAS_FAIXAS.items():
    quantis = desenvolvimento_contexto.select(
        pl.col(coluna).quantile(0.25).alias('p25'),
        pl.col(coluna).quantile(0.75).alias('p75'),
    ).collect().row(0, named=True)
    cortes[nome] = (float(quantis['p25']), float(quantis['p75']))

display(pl.DataFrame([
    {'indicador': nome, 'limite_grupo_menor': baixo, 'limite_grupo_maior': alto}
    for nome, (baixo, alto) in cortes.items()
]))

indicador,limite_grupo_menor,limite_grupo_maior
str,f64,f64
"""energia""",0.993915,1.0
"""conectividade""",0.888889,0.983192
"""saneamento""",0.535714,0.95288
"""biblioteca""",0.454545,0.742268
"""laboratorio""",0.230137,0.440341
"""quadra""",0.333333,0.555556
"""infraestrutura""",0.641667,0.799686
"""inse""",4.555143,5.270837


In [3]:
RESULTADOS_TESTES = []


def resumir_grupo(frame, coluna_contexto=None):
    expressoes = [
        pl.len().alias('registros'),
        pl.col(TARGET).mean().alias('taxa_observada'),
        pl.col('probabilidade_modelo').mean().alias('probabilidade_media'),
        pl.col('classe_modelo').mean().alias('taxa_classificada_positiva'),
    ]
    if coluna_contexto is not None:
        expressoes.append(pl.col(coluna_contexto).mean().alias('media_indicador'))
    return frame.select(expressoes).row(0, named=True)


def comparar_grupos_reais(nome_teste, nome_a, grupo_a, nome_b, grupo_b, caracteristica, coluna_contexto=None):
    a = resumir_grupo(grupo_a, coluna_contexto)
    b = resumir_grupo(grupo_b, coluna_contexto)
    if min(a['registros'], b['registros']) < 500:
        raise ValueError(f'{nome_teste}: um dos grupos possui menos de 500 registros.')

    resultado = {
        'teste': nome_teste,
        'cenario_a': nome_a,
        'registros_a': a['registros'],
        'observado_a': a['taxa_observada'],
        'previsto_a': a['probabilidade_media'],
        'cenario_b': nome_b,
        'registros_b': b['registros'],
        'observado_b': b['taxa_observada'],
        'previsto_b': b['probabilidade_media'],
        'diferenca_observada_pp': (b['taxa_observada'] - a['taxa_observada']) * 100,
        'diferenca_prevista_pp': (b['probabilidade_media'] - a['probabilidade_media']) * 100,
        'media_indicador_a': a.get('media_indicador'),
        'media_indicador_b': b.get('media_indicador'),
        'caracteristica': caracteristica,
    }

    # Permite reexecutar uma célula sem duplicar a linha do resumo.
    RESULTADOS_TESTES[:] = [r for r in RESULTADOS_TESTES if r['teste'] != nome_teste]
    RESULTADOS_TESTES.append(resultado)

    print('RESULTADO TÉCNICO — GRUPOS REAIS DO TESTE')
    print(f'• Grupo A — {nome_a}: {a["registros"]:,} registros')
    if coluna_contexto is not None:
        print(f'  Média do indicador: {a["media_indicador"]:.1%}' if 0 <= a['media_indicador'] <= 1 else f'  Média do indicador: {a["media_indicador"]:.3f}')
    print(f'  Alfabetização observada: {a["taxa_observada"]:.1%}')
    print(f'  Estimativa média do modelo: {a["probabilidade_media"]:.1%}')
    print(f'• Grupo B — {nome_b}: {b["registros"]:,} registros')
    if coluna_contexto is not None:
        print(f'  Média do indicador: {b["media_indicador"]:.1%}' if 0 <= b['media_indicador'] <= 1 else f'  Média do indicador: {b["media_indicador"]:.3f}')
    print(f'  Alfabetização observada: {b["taxa_observada"]:.1%}')
    print(f'  Estimativa média do modelo: {b["probabilidade_media"]:.1%}')
    print(f'• Diferença observada B menos A: {resultado["diferenca_observada_pp"]:+.1f} pontos percentuais')
    print(f'• Diferença prevista B menos A: {resultado["diferenca_prevista_pp"]:+.1f} pontos percentuais')
    print(f'• Referência global: acurácia {METRICAS_OFICIAIS["accuracy"]:.1%} | AUC-ROC {METRICAS_OFICIAIS["roc_auc"]:.1%}')
    return resultado


def interpretar_para_gestao(resultado):
    obs_a, obs_b = resultado['observado_a'], resultado['observado_b']
    prev_a, prev_b = resultado['previsto_a'], resultado['previsto_b']
    delta_obs = resultado['diferenca_observada_pp']
    delta_prev = resultado['diferenca_prevista_pp']
    mesma_direcao = np.sign(delta_obs) == np.sign(delta_prev) or abs(delta_obs) < 0.05 or abs(delta_prev) < 0.05

    print('INTERPRETAÇÃO PARA TOMADA DE DECISÃO')
    print(
        f'Na base real de teste, “{resultado["cenario_a"]}” apresentou {obs_a:.1%} de alfabetização observada, '
        f'enquanto “{resultado["cenario_b"]}” apresentou {obs_b:.1%}.'
    )
    print(
        f'O modelo estimou, em média, {prev_a:.1%} para o primeiro grupo e {prev_b:.1%} para o segundo.'
    )
    print(
        f'A diferença observada foi {delta_obs:+.1f} pontos percentuais e a diferença prevista pelo modelo foi '
        f'{delta_prev:+.1f} pontos percentuais.'
    )
    if mesma_direcao:
        print('O modelo reproduziu a direção geral da diferença observada entre os grupos.')
    else:
        print('A direção prevista pelo modelo divergiu da diferença observada; o resultado exige cautela adicional.')
    print(
        'Os grupos são reais, mas também podem diferir em outras características. Portanto, a comparação descreve '
        'uma associação e não comprova que a característica analisada causou a diferença.'
    )

## Teste 1

### Cobertura municipal de energia

Comparamos municípios nos grupos de menor e maior proporção de escolas atendidas pela rede pública de energia.

### Resultado técnico

A seguir estão os resultados calculados com registros reais do teste oficial.

In [4]:
baixo, alto = cortes['energia']
resultado_energia = comparar_grupos_reais(
    'Cobertura municipal de energia',
    'Menor cobertura de energia', avaliacao.filter(pl.col('in_energia_rede_publica') <= baixo),
    'Maior cobertura de energia', avaliacao.filter(pl.col('in_energia_rede_publica') >= alto),
    'proporção municipal de escolas com energia da rede pública',
    'in_energia_rede_publica',
)

RESULTADO TÉCNICO — GRUPOS REAIS DO TESTE
• Grupo A — Menor cobertura de energia: 148,493 registros
  Média do indicador: 93.0%
  Alfabetização observada: 47.1%
  Estimativa média do modelo: 47.4%
• Grupo B — Maior cobertura de energia: 342,056 registros
  Média do indicador: 100.0%
  Alfabetização observada: 54.0%
  Estimativa média do modelo: 53.8%
• Diferença observada B menos A: +6.9 pontos percentuais
• Diferença prevista B menos A: +6.4 pontos percentuais
• Referência global: acurácia 62.1% | AUC-ROC 67.0%


### Interpretação para tomada de decisão

A célula seguinte traduz o resultado, compara previsão e realidade e registra o limite da conclusão.

In [5]:
interpretar_para_gestao(resultado_energia)

INTERPRETAÇÃO PARA TOMADA DE DECISÃO
Na base real de teste, “Menor cobertura de energia” apresentou 47.1% de alfabetização observada, enquanto “Maior cobertura de energia” apresentou 54.0%.
O modelo estimou, em média, 47.4% para o primeiro grupo e 53.8% para o segundo.
A diferença observada foi +6.9 pontos percentuais e a diferença prevista pelo modelo foi +6.4 pontos percentuais.
O modelo reproduziu a direção geral da diferença observada entre os grupos.
Os grupos são reais, mas também podem diferir em outras características. Portanto, a comparação descreve uma associação e não comprova que a característica analisada causou a diferença.


[Voltar ao menu](#Menu)

## Teste 2

### Cobertura municipal de conectividade

O indicador combina as proporções municipais de escolas com internet e com banda larga.

### Resultado técnico

A seguir estão os resultados calculados com registros reais do teste oficial.

In [6]:
baixo, alto = cortes['conectividade']
resultado_internet = comparar_grupos_reais(
    'Cobertura municipal de conectividade',
    'Menor cobertura de conectividade', avaliacao.filter(pl.col('indice_conectividade') <= baixo),
    'Maior cobertura de conectividade', avaliacao.filter(pl.col('indice_conectividade') >= alto),
    'média municipal de cobertura de internet e banda larga',
    'indice_conectividade',
)

RESULTADO TÉCNICO — GRUPOS REAIS DO TESTE
• Grupo A — Menor cobertura de conectividade: 145,734 registros
  Média do indicador: 76.2%
  Alfabetização observada: 49.8%
  Estimativa média do modelo: 49.7%
• Grupo B — Maior cobertura de conectividade: 145,084 registros
  Média do indicador: 99.2%
  Alfabetização observada: 54.2%
  Estimativa média do modelo: 54.1%
• Diferença observada B menos A: +4.4 pontos percentuais
• Diferença prevista B menos A: +4.3 pontos percentuais
• Referência global: acurácia 62.1% | AUC-ROC 67.0%


### Interpretação para tomada de decisão

A célula seguinte traduz o resultado, compara previsão e realidade e registra o limite da conclusão.

In [7]:
interpretar_para_gestao(resultado_internet)

INTERPRETAÇÃO PARA TOMADA DE DECISÃO
Na base real de teste, “Menor cobertura de conectividade” apresentou 49.8% de alfabetização observada, enquanto “Maior cobertura de conectividade” apresentou 54.2%.
O modelo estimou, em média, 49.7% para o primeiro grupo e 54.1% para o segundo.
A diferença observada foi +4.4 pontos percentuais e a diferença prevista pelo modelo foi +4.3 pontos percentuais.
O modelo reproduziu a direção geral da diferença observada entre os grupos.
Os grupos são reais, mas também podem diferir em outras características. Portanto, a comparação descreve uma associação e não comprova que a característica analisada causou a diferença.


[Voltar ao menu](#Menu)

## Teste 3

### Cobertura municipal de água e esgoto

O indicador combina as proporções municipais de escolas com água e esgoto da rede pública.

### Resultado técnico

A seguir estão os resultados calculados com registros reais do teste oficial.

In [8]:
baixo, alto = cortes['saneamento']
resultado_saneamento = comparar_grupos_reais(
    'Cobertura municipal de água e esgoto',
    'Menor cobertura de água e esgoto', avaliacao.filter(pl.col('indice_saneamento') <= baixo),
    'Maior cobertura de água e esgoto', avaliacao.filter(pl.col('indice_saneamento') >= alto),
    'média municipal de cobertura de água e esgoto',
    'indice_saneamento',
)

RESULTADO TÉCNICO — GRUPOS REAIS DO TESTE
• Grupo A — Menor cobertura de água e esgoto: 145,340 registros
  Média do indicador: 32.7%
  Alfabetização observada: 49.5%
  Estimativa média do modelo: 49.5%
• Grupo B — Maior cobertura de água e esgoto: 145,658 registros
  Média do indicador: 98.6%
  Alfabetização observada: 51.4%
  Estimativa média do modelo: 51.6%
• Diferença observada B menos A: +1.9 pontos percentuais
• Diferença prevista B menos A: +2.1 pontos percentuais
• Referência global: acurácia 62.1% | AUC-ROC 67.0%


### Interpretação para tomada de decisão

A célula seguinte traduz o resultado, compara previsão e realidade e registra o limite da conclusão.

In [9]:
interpretar_para_gestao(resultado_saneamento)

INTERPRETAÇÃO PARA TOMADA DE DECISÃO
Na base real de teste, “Menor cobertura de água e esgoto” apresentou 49.5% de alfabetização observada, enquanto “Maior cobertura de água e esgoto” apresentou 51.4%.
O modelo estimou, em média, 49.5% para o primeiro grupo e 51.6% para o segundo.
A diferença observada foi +1.9 pontos percentuais e a diferença prevista pelo modelo foi +2.1 pontos percentuais.
O modelo reproduziu a direção geral da diferença observada entre os grupos.
Os grupos são reais, mas também podem diferir em outras características. Portanto, a comparação descreve uma associação e não comprova que a característica analisada causou a diferença.


[Voltar ao menu](#Menu)

## Teste 4

### Cobertura municipal de biblioteca ou sala de leitura

Comparamos grupos reais de municípios com menor e maior proporção de escolas com biblioteca ou sala de leitura.

### Resultado técnico

A seguir estão os resultados calculados com registros reais do teste oficial.

In [10]:
baixo, alto = cortes['biblioteca']
resultado_biblioteca = comparar_grupos_reais(
    'Cobertura municipal de biblioteca',
    'Menor cobertura de biblioteca', avaliacao.filter(pl.col('in_biblioteca_sala_leitura') <= baixo),
    'Maior cobertura de biblioteca', avaliacao.filter(pl.col('in_biblioteca_sala_leitura') >= alto),
    'proporção municipal de escolas com biblioteca ou sala de leitura',
    'in_biblioteca_sala_leitura',
)

RESULTADO TÉCNICO — GRUPOS REAIS DO TESTE
• Grupo A — Menor cobertura de biblioteca: 145,948 registros
  Média do indicador: 30.9%
  Alfabetização observada: 47.4%
  Estimativa média do modelo: 47.5%
• Grupo B — Maior cobertura de biblioteca: 146,118 registros
  Média do indicador: 82.1%
  Alfabetização observada: 54.5%
  Estimativa média do modelo: 54.4%
• Diferença observada B menos A: +7.0 pontos percentuais
• Diferença prevista B menos A: +6.9 pontos percentuais
• Referência global: acurácia 62.1% | AUC-ROC 67.0%


### Interpretação para tomada de decisão

A célula seguinte traduz o resultado, compara previsão e realidade e registra o limite da conclusão.

In [11]:
interpretar_para_gestao(resultado_biblioteca)

INTERPRETAÇÃO PARA TOMADA DE DECISÃO
Na base real de teste, “Menor cobertura de biblioteca” apresentou 47.4% de alfabetização observada, enquanto “Maior cobertura de biblioteca” apresentou 54.5%.
O modelo estimou, em média, 47.5% para o primeiro grupo e 54.4% para o segundo.
A diferença observada foi +7.0 pontos percentuais e a diferença prevista pelo modelo foi +6.9 pontos percentuais.
O modelo reproduziu a direção geral da diferença observada entre os grupos.
Os grupos são reais, mas também podem diferir em outras características. Portanto, a comparação descreve uma associação e não comprova que a característica analisada causou a diferença.


[Voltar ao menu](#Menu)

## Teste 5

### Cobertura municipal de laboratório de informática

Comparamos grupos reais de municípios com menor e maior proporção de escolas com laboratório de informática.

### Resultado técnico

A seguir estão os resultados calculados com registros reais do teste oficial.

In [12]:
baixo, alto = cortes['laboratorio']
resultado_laboratorio = comparar_grupos_reais(
    'Cobertura municipal de laboratório',
    'Menor cobertura de laboratório', avaliacao.filter(pl.col('in_laboratorio_informatica') <= baixo),
    'Maior cobertura de laboratório', avaliacao.filter(pl.col('in_laboratorio_informatica') >= alto),
    'proporção municipal de escolas com laboratório de informática',
    'in_laboratorio_informatica',
)

RESULTADO TÉCNICO — GRUPOS REAIS DO TESTE
• Grupo A — Menor cobertura de laboratório: 146,779 registros
  Média do indicador: 12.7%
  Alfabetização observada: 47.1%
  Estimativa média do modelo: 47.2%
• Grupo B — Maior cobertura de laboratório: 147,239 registros
  Média do indicador: 54.0%
  Alfabetização observada: 56.9%
  Estimativa média do modelo: 56.8%
• Diferença observada B menos A: +9.8 pontos percentuais
• Diferença prevista B menos A: +9.6 pontos percentuais
• Referência global: acurácia 62.1% | AUC-ROC 67.0%


### Interpretação para tomada de decisão

A célula seguinte traduz o resultado, compara previsão e realidade e registra o limite da conclusão.

In [13]:
interpretar_para_gestao(resultado_laboratorio)

INTERPRETAÇÃO PARA TOMADA DE DECISÃO
Na base real de teste, “Menor cobertura de laboratório” apresentou 47.1% de alfabetização observada, enquanto “Maior cobertura de laboratório” apresentou 56.9%.
O modelo estimou, em média, 47.2% para o primeiro grupo e 56.8% para o segundo.
A diferença observada foi +9.8 pontos percentuais e a diferença prevista pelo modelo foi +9.6 pontos percentuais.
O modelo reproduziu a direção geral da diferença observada entre os grupos.
Os grupos são reais, mas também podem diferir em outras características. Portanto, a comparação descreve uma associação e não comprova que a característica analisada causou a diferença.


[Voltar ao menu](#Menu)

## Teste 6

### Cobertura municipal de quadra de esportes

Comparamos grupos reais de municípios com menor e maior proporção de escolas com quadra de esportes.

### Resultado técnico

A seguir estão os resultados calculados com registros reais do teste oficial.

In [14]:
baixo, alto = cortes['quadra']
resultado_quadra = comparar_grupos_reais(
    'Cobertura municipal de quadra',
    'Menor cobertura de quadra', avaliacao.filter(pl.col('in_quadra_esportes') <= baixo),
    'Maior cobertura de quadra', avaliacao.filter(pl.col('in_quadra_esportes') >= alto),
    'proporção municipal de escolas com quadra de esportes',
    'in_quadra_esportes',
)

RESULTADO TÉCNICO — GRUPOS REAIS DO TESTE
• Grupo A — Menor cobertura de quadra: 152,824 registros
  Média do indicador: 19.9%
  Alfabetização observada: 48.5%
  Estimativa média do modelo: 48.6%
• Grupo B — Maior cobertura de quadra: 147,373 registros
  Média do indicador: 62.8%
  Alfabetização observada: 56.0%
  Estimativa média do modelo: 55.9%
• Diferença observada B menos A: +7.4 pontos percentuais
• Diferença prevista B menos A: +7.3 pontos percentuais
• Referência global: acurácia 62.1% | AUC-ROC 67.0%


### Interpretação para tomada de decisão

A célula seguinte traduz o resultado, compara previsão e realidade e registra o limite da conclusão.

In [15]:
interpretar_para_gestao(resultado_quadra)

INTERPRETAÇÃO PARA TOMADA DE DECISÃO
Na base real de teste, “Menor cobertura de quadra” apresentou 48.5% de alfabetização observada, enquanto “Maior cobertura de quadra” apresentou 56.0%.
O modelo estimou, em média, 48.6% para o primeiro grupo e 55.9% para o segundo.
A diferença observada foi +7.4 pontos percentuais e a diferença prevista pelo modelo foi +7.3 pontos percentuais.
O modelo reproduziu a direção geral da diferença observada entre os grupos.
Os grupos são reais, mas também podem diferir em outras características. Portanto, a comparação descreve uma associação e não comprova que a característica analisada causou a diferença.


[Voltar ao menu](#Menu)

## Teste 7

### Índice agregado de infraestrutura municipal

O índice é a média das oito proporções municipais de infraestrutura utilizadas pelo modelo.

### Resultado técnico

A seguir estão os resultados calculados com registros reais do teste oficial.

In [16]:
baixo, alto = cortes['infraestrutura']
resultado_infraestrutura = comparar_grupos_reais(
    'Índice agregado de infraestrutura',
    'Menor infraestrutura agregada', avaliacao.filter(pl.col('indice_infraestrutura') <= baixo),
    'Maior infraestrutura agregada', avaliacao.filter(pl.col('indice_infraestrutura') >= alto),
    'índice médio das oito proporções municipais de infraestrutura',
    'indice_infraestrutura',
)

RESULTADO TÉCNICO — GRUPOS REAIS DO TESTE
• Grupo A — Menor infraestrutura agregada: 145,450 registros
  Média do indicador: 50.7%
  Alfabetização observada: 48.4%
  Estimativa média do modelo: 48.5%
• Grupo B — Maior infraestrutura agregada: 145,660 registros
  Média do indicador: 83.1%
  Alfabetização observada: 54.5%
  Estimativa média do modelo: 54.4%
• Diferença observada B menos A: +6.0 pontos percentuais
• Diferença prevista B menos A: +5.9 pontos percentuais
• Referência global: acurácia 62.1% | AUC-ROC 67.0%


### Interpretação para tomada de decisão

A célula seguinte traduz o resultado, compara previsão e realidade e registra o limite da conclusão.

In [17]:
interpretar_para_gestao(resultado_infraestrutura)

INTERPRETAÇÃO PARA TOMADA DE DECISÃO
Na base real de teste, “Menor infraestrutura agregada” apresentou 48.4% de alfabetização observada, enquanto “Maior infraestrutura agregada” apresentou 54.5%.
O modelo estimou, em média, 48.5% para o primeiro grupo e 54.4% para o segundo.
A diferença observada foi +6.0 pontos percentuais e a diferença prevista pelo modelo foi +5.9 pontos percentuais.
O modelo reproduziu a direção geral da diferença observada entre os grupos.
Os grupos são reais, mas também podem diferir em outras características. Portanto, a comparação descreve uma associação e não comprova que a característica analisada causou a diferença.


[Voltar ao menu](#Menu)

## Teste 8

### INSE médio

Comparamos registros reais nos grupos inferior e superior do Indicador de Nível Socioeconômico médio. As variáveis financeiras não são misturadas neste contraste.

### Resultado técnico

A seguir estão os resultados calculados com registros reais do teste oficial.

In [18]:
baixo, alto = cortes['inse']
resultado_inse = comparar_grupos_reais(
    'INSE médio',
    'INSE no grupo inferior', avaliacao.filter(pl.col('media_inse') <= baixo),
    'INSE no grupo superior', avaliacao.filter(pl.col('media_inse') >= alto),
    'INSE médio do contexto educacional',
    'media_inse',
)

RESULTADO TÉCNICO — GRUPOS REAIS DO TESTE
• Grupo A — INSE no grupo inferior: 145,769 registros
  Média do indicador: 4.334
  Alfabetização observada: 48.7%
  Estimativa média do modelo: 48.9%
• Grupo B — INSE no grupo superior: 149,037 registros
  Média do indicador: 5.422
  Alfabetização observada: 54.3%
  Estimativa média do modelo: 54.1%
• Diferença observada B menos A: +5.6 pontos percentuais
• Diferença prevista B menos A: +5.2 pontos percentuais
• Referência global: acurácia 62.1% | AUC-ROC 67.0%


### Interpretação para tomada de decisão

A célula seguinte traduz o resultado, compara previsão e realidade e registra o limite da conclusão.

In [19]:
interpretar_para_gestao(resultado_inse)

INTERPRETAÇÃO PARA TOMADA DE DECISÃO
Na base real de teste, “INSE no grupo inferior” apresentou 48.7% de alfabetização observada, enquanto “INSE no grupo superior” apresentou 54.3%.
O modelo estimou, em média, 48.9% para o primeiro grupo e 54.1% para o segundo.
A diferença observada foi +5.6 pontos percentuais e a diferença prevista pelo modelo foi +5.2 pontos percentuais.
O modelo reproduziu a direção geral da diferença observada entre os grupos.
Os grupos são reais, mas também podem diferir em outras características. Portanto, a comparação descreve uma associação e não comprova que a característica analisada causou a diferença.


[Voltar ao menu](#Menu)

## Teste 9

### Contextos observados em São Paulo e Ceará

Comparamos todos os registros reais de SP e CE presentes no teste. Não mantemos as demais variáveis artificialmente fixas; a diferença resume contextos territoriais distintos.

### Resultado técnico

A seguir estão os resultados calculados com registros reais do teste oficial.

In [20]:
resultado_uf = comparar_grupos_reais(
    'Contextos observados por UF',
    'Registros observados em SP', avaliacao.filter(pl.col('sigla_uf') == 'SP'),
    'Registros observados em CE', avaliacao.filter(pl.col('sigla_uf') == 'CE'),
    'contexto territorial agregado representado pela UF',
)

RESULTADO TÉCNICO — GRUPOS REAIS DO TESTE
• Grupo A — Registros observados em SP: 66,925 registros
  Alfabetização observada: 52.2%
  Estimativa média do modelo: 52.2%
• Grupo B — Registros observados em CE: 28,799 registros
  Alfabetização observada: 81.8%
  Estimativa média do modelo: 81.7%
• Diferença observada B menos A: +29.6 pontos percentuais
• Diferença prevista B menos A: +29.4 pontos percentuais
• Referência global: acurácia 62.1% | AUC-ROC 67.0%


### Interpretação para tomada de decisão

A célula seguinte traduz o resultado, compara previsão e realidade e registra o limite da conclusão.

In [21]:
interpretar_para_gestao(resultado_uf)

INTERPRETAÇÃO PARA TOMADA DE DECISÃO
Na base real de teste, “Registros observados em SP” apresentou 52.2% de alfabetização observada, enquanto “Registros observados em CE” apresentou 81.8%.
O modelo estimou, em média, 52.2% para o primeiro grupo e 81.7% para o segundo.
A diferença observada foi +29.6 pontos percentuais e a diferença prevista pelo modelo foi +29.4 pontos percentuais.
O modelo reproduziu a direção geral da diferença observada entre os grupos.
Os grupos são reais, mas também podem diferir em outras características. Portanto, a comparação descreve uma associação e não comprova que a característica analisada causou a diferença.


[Voltar ao menu](#Menu)

## Teste 10

### Redes estadual e municipal

Comparamos todos os registros reais das redes estadual e municipal presentes no teste. As redes podem diferir em território, infraestrutura e composição socioeconômica.

### Resultado técnico

A seguir estão os resultados calculados com registros reais do teste oficial.

In [22]:
resultado_rede = comparar_grupos_reais(
    'Rede de ensino',
    'Rede estadual', avaliacao.filter(pl.col('rede') == '2'),
    'Rede municipal', avaliacao.filter(pl.col('rede') == '3'),
    'rede de ensino observada',
)

RESULTADO TÉCNICO — GRUPOS REAIS DO TESTE
• Grupo A — Rede estadual: 65,599 registros
  Alfabetização observada: 53.0%
  Estimativa média do modelo: 52.3%
• Grupo B — Rede municipal: 515,992 registros
  Alfabetização observada: 51.2%
  Estimativa média do modelo: 51.3%
• Diferença observada B menos A: -1.8 pontos percentuais
• Diferença prevista B menos A: -1.1 pontos percentuais
• Referência global: acurácia 62.1% | AUC-ROC 67.0%


### Interpretação para tomada de decisão

A célula seguinte traduz o resultado, compara previsão e realidade e registra o limite da conclusão.

In [23]:
interpretar_para_gestao(resultado_rede)

INTERPRETAÇÃO PARA TOMADA DE DECISÃO
Na base real de teste, “Rede estadual” apresentou 53.0% de alfabetização observada, enquanto “Rede municipal” apresentou 51.2%.
O modelo estimou, em média, 52.3% para o primeiro grupo e 51.3% para o segundo.
A diferença observada foi -1.8 pontos percentuais e a diferença prevista pelo modelo foi -1.1 pontos percentuais.
O modelo reproduziu a direção geral da diferença observada entre os grupos.
Os grupos são reais, mas também podem diferir em outras características. Portanto, a comparação descreve uma associação e não comprova que a característica analisada causou a diferença.


[Voltar ao menu](#Menu)

## Resumo

### Resumo executivo

A tabela reúne o tamanho dos grupos, a alfabetização observada e a estimativa média do modelo. Diferenças positivas indicam valor maior no grupo B; diferenças negativas indicam valor menor.

In [24]:
resumo = pl.DataFrame(RESULTADOS_TESTES).select(
    'teste', 'cenario_a', 'registros_a', 'observado_a', 'previsto_a',
    'cenario_b', 'registros_b', 'observado_b', 'previsto_b',
    'diferenca_observada_pp', 'diferenca_prevista_pp',
)

resumo_exibicao = resumo.with_columns(
    pl.col('observado_a').map_elements(lambda x: f'{x:.1%}', return_dtype=pl.String),
    pl.col('previsto_a').map_elements(lambda x: f'{x:.1%}', return_dtype=pl.String),
    pl.col('observado_b').map_elements(lambda x: f'{x:.1%}', return_dtype=pl.String),
    pl.col('previsto_b').map_elements(lambda x: f'{x:.1%}', return_dtype=pl.String),
    pl.col('diferenca_observada_pp').map_elements(lambda x: f'{x:+.1f} pp', return_dtype=pl.String),
    pl.col('diferenca_prevista_pp').map_elements(lambda x: f'{x:+.1f} pp', return_dtype=pl.String),
)
display(resumo_exibicao)

teste,cenario_a,registros_a,observado_a,previsto_a,cenario_b,registros_b,observado_b,previsto_b,diferenca_observada_pp,diferenca_prevista_pp
str,str,i64,str,str,str,i64,str,str,str,str
"""Cobertura municipal de energia""","""Menor cobertura de energia""",148493,"""47.1%""","""47.4%""","""Maior cobertura de energia""",342056,"""54.0%""","""53.8%""","""+6.9 pp""","""+6.4 pp"""
"""Cobertura municipal de conecti…","""Menor cobertura de conectivida…",145734,"""49.8%""","""49.7%""","""Maior cobertura de conectivida…",145084,"""54.2%""","""54.1%""","""+4.4 pp""","""+4.3 pp"""
"""Cobertura municipal de água e …","""Menor cobertura de água e esgo…",145340,"""49.5%""","""49.5%""","""Maior cobertura de água e esgo…",145658,"""51.4%""","""51.6%""","""+1.9 pp""","""+2.1 pp"""
"""Cobertura municipal de bibliot…","""Menor cobertura de biblioteca""",145948,"""47.4%""","""47.5%""","""Maior cobertura de biblioteca""",146118,"""54.5%""","""54.4%""","""+7.0 pp""","""+6.9 pp"""
"""Cobertura municipal de laborat…","""Menor cobertura de laboratório""",146779,"""47.1%""","""47.2%""","""Maior cobertura de laboratório""",147239,"""56.9%""","""56.8%""","""+9.8 pp""","""+9.6 pp"""
"""Cobertura municipal de quadra""","""Menor cobertura de quadra""",152824,"""48.5%""","""48.6%""","""Maior cobertura de quadra""",147373,"""56.0%""","""55.9%""","""+7.4 pp""","""+7.3 pp"""
"""Índice agregado de infraestrut…","""Menor infraestrutura agregada""",145450,"""48.4%""","""48.5%""","""Maior infraestrutura agregada""",145660,"""54.5%""","""54.4%""","""+6.0 pp""","""+5.9 pp"""
"""INSE médio""","""INSE no grupo inferior""",145769,"""48.7%""","""48.9%""","""INSE no grupo superior""",149037,"""54.3%""","""54.1%""","""+5.6 pp""","""+5.2 pp"""
"""Contextos observados por UF""","""Registros observados em SP""",66925,"""52.2%""","""52.2%""","""Registros observados em CE""",28799,"""81.8%""","""81.7%""","""+29.6 pp""","""+29.4 pp"""


## Analise final UF

### Peso da variável UF no modelo

O teste entre SP e CE mostrou uma diferença territorial expressiva. Para avaliar se isso é apenas uma diferença observada entre os grupos ou também uma dependência interna do modelo, usamos duas verificações publicadas pelo pipeline oficial:

1. **importância por permutação:** mede quanto o desempenho do modelo treinado cai quando a informação de UF é embaralhada;
2. **análise com e sem UF:** retreina o mesmo Random Forest sem `sigla_uf` e compara o desempenho no teste.

### Resultado técnico

Os valores abaixo são lidos diretamente das tabelas oficiais produzidas pelo pipeline completo.

In [25]:
IMPORTANCIA_PATH = PROJECT_ROOT / 'resultados' / 'tabelas' / 'importancia_permutacao.csv'
SENSIBILIDADE_UF_PATH = PROJECT_ROOT / 'resultados' / 'tabelas' / '18_sensibilidade_sigla_uf.csv'

importancias_oficiais = pl.read_csv(IMPORTANCIA_PATH).with_row_index('posicao', offset=1)
sensibilidade_uf = pl.read_csv(SENSIBILIDADE_UF_PATH)

linha_uf = importancias_oficiais.filter(pl.col('feature') == 'sigla_uf').row(0, named=True)
linha_auc = sensibilidade_uf.filter(pl.col('metrica') == 'roc_auc').row(0, named=True)
linha_ap = sensibilidade_uf.filter(pl.col('metrica') == 'average_precision').row(0, named=True)

print('RESULTADO TÉCNICO — DEPENDÊNCIA TERRITORIAL')
print(f'• Posição de sigla_uf na importância por permutação: {linha_uf["posicao"]}º de {importancias_oficiais.height} variáveis')
print(f'• Redução média de AUC ao embaralhar UF: {linha_uf["importancia_media"] * 100:.2f} pontos')
print(f'• Desvio entre repetições: {linha_uf["importancia_desvio"] * 100:.2f} ponto')
print(f'• AUC do modelo oficial com UF: {linha_auc["com_sigla_uf"]:.4f}')
print(f'• AUC do modelo retreinado sem UF: {linha_auc["sem_sigla_uf"]:.4f}')
print(f'• Diferença com menos sem UF: {linha_auc["diferenca_com_menos_sem"] * 100:+.2f} ponto percentual')
print(f'• Average Precision com UF: {linha_ap["com_sigla_uf"]:.4f}')
print(f'• Average Precision sem UF: {linha_ap["sem_sigla_uf"]:.4f}')

display(importancias_oficiais.head(8))
display(sensibilidade_uf)

RESULTADO TÉCNICO — DEPENDÊNCIA TERRITORIAL
• Posição de sigla_uf na importância por permutação: 1º de 18 variáveis
• Redução média de AUC ao embaralhar UF: 8.15 pontos
• Desvio entre repetições: 0.17 ponto
• AUC do modelo oficial com UF: 0.6705
• AUC do modelo retreinado sem UF: 0.6757
• Diferença com menos sem UF: -0.52 ponto percentual
• Average Precision com UF: 0.6841
• Average Precision sem UF: 0.6910


posicao,feature,importancia_media,importancia_desvio
u32,str,f64,f64
1,"""sigla_uf""",0.081454,0.001723
2,"""valor_bolsa_familia__signed_lo…",0.013114,0.000664
3,"""media_inse""",0.008716,0.000608
4,"""rede""",0.007759,0.000488
5,"""in_laboratorio_informatica""",0.00525,0.000376
6,"""valor_receitas__signed_log1p""",0.004941,0.000459
7,"""in_biblioteca_sala_leitura""",0.0047,0.000729
8,"""in_internet""",0.004555,0.000394


metrica,com_sigla_uf,sem_sigla_uf,diferenca_com_menos_sem
str,f64,f64,f64
"""roc_auc""",0.670483,0.675669,-0.005187
"""average_precision""",0.684097,0.691022,-0.006925
"""balanced_accuracy""",0.619615,0.623244,-0.003629
"""f1""",0.639195,0.636279,0.002916


### Interpretação para tomada de decisão

A célula seguinte traduz conjuntamente as duas evidências. É importante não confundir “o modelo usa muito a UF” com “a UF causa alfabetização” ou “a UF melhora o modelo”.

In [26]:
queda_permutacao_pp = linha_uf['importancia_media'] * 100
ganho_sem_uf_pp = (linha_auc['sem_sigla_uf'] - linha_auc['com_sigla_uf']) * 100

print('INTERPRETAÇÃO FINAL PARA GESTORES')
print(
    f'A UF é a variável da qual o modelo treinado mais depende: quando essa informação é embaralhada, '
    f'a capacidade de ordenação do modelo cai aproximadamente {queda_permutacao_pp:.1f} pontos de AUC.'
)
print(
    f'Entretanto, quando o modelo é treinado novamente sem a UF, a AUC aumenta cerca de '
    f'{ganho_sem_uf_pp:.1f} ponto percentual. Portanto, forte dependência não significa ganho de qualidade.'
)
print(
    'A variável UF resume diferenças de políticas educacionais, investimento, infraestrutura, composição '
    'socioeconômica e cobertura dos dados. Ela funciona como uma proxy territorial, e não como causa isolada.'
)
print(
    'A diferença observada entre SP e CE deve ser usada para orientar investigação territorial e monitoramento '
    'de equidade. Ela não deve, sozinha, justificar decisões sobre alunos nem afirmar superioridade de uma UF.'
)
print(
    'Como análise de robustez, o resultado sugere acompanhar também a versão sem UF ou, no mínimo, reportar '
    'métricas por UF antes de qualquer uso operacional do modelo.'
)

INTERPRETAÇÃO FINAL PARA GESTORES
A UF é a variável da qual o modelo treinado mais depende: quando essa informação é embaralhada, a capacidade de ordenação do modelo cai aproximadamente 8.1 pontos de AUC.
Entretanto, quando o modelo é treinado novamente sem a UF, a AUC aumenta cerca de 0.5 ponto percentual. Portanto, forte dependência não significa ganho de qualidade.
A variável UF resume diferenças de políticas educacionais, investimento, infraestrutura, composição socioeconômica e cobertura dos dados. Ela funciona como uma proxy territorial, e não como causa isolada.
A diferença observada entre SP e CE deve ser usada para orientar investigação territorial e monitoramento de equidade. Ela não deve, sozinha, justificar decisões sobre alunos nem afirmar superioridade de uma UF.
Como análise de robustez, o resultado sugere acompanhar também a versão sem UF ou, no mínimo, reportar métricas por UF antes de qualquer uso operacional do modelo.


[Voltar ao menu](#Menu)

## Guia final

### Interpretação para gestores

### Forma recomendada de apresentar

> A análise utiliza grupos que realmente existem na base oficial de teste. Para cada grupo, mostramos a proporção observada de alfabetização e a estimativa média produzida pelo modelo. As diferenças ajudam a identificar padrões e prioridades de investigação, mas não provam relações de causa e efeito.

### O que cada número significa

- **Registros:** quantidade de observações reais que sustentam o resultado;
- **Alfabetização observada:** proporção que efetivamente aparece como alfabetizada na base;
- **Estimativa média do modelo:** média das probabilidades atribuídas aos registros do grupo;
- **Diferença observada:** distância real entre as taxas dos dois grupos;
- **Diferença prevista:** distância que o modelo conseguiu representar;
- **Acurácia e AUC-ROC:** desempenho global do modelo, não confiança individual.

### Limites importantes

- As variáveis de infraestrutura são proporções municipais, não características confirmadas da escola frequentada por cada aluno;
- grupos de maior e menor cobertura também podem diferir em território, renda, rede e outras condições;
- UF funciona como proxy de múltiplas diferenças regionais;
- as análises são adequadas para compreensão agregada e apoio à investigação, não para decisão automática sobre uma criança.

[Voltar ao menu](#Menu)